<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2><h2 style="color:Green">Review</h2>

----

### Вариант задания 18


Создать базовый класс Review в C#, который будет представлять отзывы о 
продуктах или услугах. На основе этого класса разработать 2-3 производных класса, 
демонстрирующих принципы наследования и полиморфизма. В каждом из классов 
должны быть реализованы новые атрибуты и методы, а также переопределены 
некоторые методы базового класса для демонстрации полиморфизма. 
Требования к базовому классу Review: 
• Атрибуты: ID отзыва (ReviewId), Текст отзыва (Text), Рейтинг (Rating). 
• Методы: 
o DisplayReview(): метод для отображения отзыва. 
o RateProduct(): метод для присвоения рейтинга продукту. 
o GetReviewDetails(): метод для получения деталей отзыва. 
Требования к производным классам: 
1. ОтзывОбслуживания (ServiceReview): Должен содержать дополнительные 
атрибуты, 
такие 
как 
Дата 
посещения 
(VisitDate). 
Метод DisplayReview() должен быть переопределен для включения даты 
посещения в отображение отзыва. 
2. ОтзывТовара (ProductReview): Должен содержать дополнительные атрибуты, 
такие как Идентификатор продукта (ProductId). Метод RateProduct() должен 
быть переопределен для связывания рейтинга с конкретным продуктом. 
3. ОтзывУслуги (ServiceReview) (если требуется третий класс): Должен 
содержать дополнительные атрибуты, такие как Время начала услуги 
(StartTime). Метод GetReviewDetails() должен быть переопределен для 
отображения времени начала услуги вместе с другими деталями отзыва.


#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [11]:
using System;
using System.Collections.Generic;
using System.Linq;

// ДЕЛЕГАТЫ
public delegate void ReviewEventHandler(object sender, ReviewEventArgs e);
public delegate void ReviewModerationHandler(Review review, bool approved);
public delegate void ReviewSearchHandler(string searchTerm, List<Review> results);

// АРГУМЕНТЫ СОБЫТИЙ
public class ReviewEventArgs : EventArgs
{
    public Review Review { get; }
    public string Action { get; }
    public DateTime Timestamp { get; }

    public ReviewEventArgs(Review review, string action)
    {
        Review = review;
        Action = action;
        Timestamp = DateTime.Now;
    }
}

public interface IReviewService
{
    void PublishReview(Review review);
    void ModerateReview(Review review);
    List<Review> GetReviewsByAuthor(string author);
    List<Review> GetPopularReviews(int minLikes);
    
    // НОВЫЕ МЕТОДЫ
    Dictionary<string, int> GetAuthorStats();
    HashSet<string> GetAllTags();
    List<Review> SearchReviews(string searchTerm);
}

public class ReviewService : IReviewService
{
    private List<Review> _allReviews = new List<Review>();
    private Dictionary<string, List<Review>> _authorReviews = new Dictionary<string, List<Review>>();
    private HashSet<string> _allTags = new HashSet<string>();
    
    // СОБЫТИЯ
    public event ReviewEventHandler ReviewPublished;
    public event ReviewEventHandler ReviewModerated;
    public event ReviewSearchHandler ReviewSearched;
    
    protected virtual void OnReviewPublished(Review review)
    {
        ReviewPublished?.Invoke(this, new ReviewEventArgs(review, "published"));
    }
    
    protected virtual void OnReviewModerated(Review review)
    {
        ReviewModerated?.Invoke(this, new ReviewEventArgs(review, "moderated"));
    }
    
    protected virtual void OnReviewSearched(string searchTerm, List<Review> results)
    {
        ReviewSearched?.Invoke(searchTerm, results);
    }
    
    public void PublishReview(Review review)
    {
        _allReviews.Add(review);
        
        // Обновляем коллекцию отзывов по авторам
        if (!_authorReviews.ContainsKey(review.Author))
        {
            _authorReviews[review.Author] = new List<Review>();
        }
        _authorReviews[review.Author].Add(review);
        
        // Обновляем коллекцию тегов
        foreach (var tag in review.Tags)
        {
            _allTags.Add(tag.ToLower());
        }
        
        Console.WriteLine($"Отзыв {review.ReviewId} опубликован в системе");
        OnReviewPublished(review);
    }
    
    public void ModerateReview(Review review)
    {
        review.VerifyReview();
        Console.WriteLine($"Отзыв {review.ReviewId} прошел модерацию");
        OnReviewModerated(review);
    }
    
    public List<Review> GetReviewsByAuthor(string author)
    {
        return _allReviews.FindAll(r => r.Author.Equals(author, StringComparison.OrdinalIgnoreCase));
    }
    
    public List<Review> GetPopularReviews(int minLikes)
    {
        return _allReviews.FindAll(r => r.Likes >= minLikes);
    }
    
    // НОВЫЕ МЕТОДЫ
    public Dictionary<string, int> GetAuthorStats()
    {
        var stats = new Dictionary<string, int>();
        foreach (var author in _authorReviews.Keys)
        {
            stats[author] = _authorReviews[author].Count;
        }
        return stats;
    }
    
    public HashSet<string> GetAllTags()
    {
        return new HashSet<string>(_allTags);
    }
    
    public List<Review> SearchReviews(string searchTerm)
    {
        var results = _allReviews.FindAll(r => 
            r.Text.Contains(searchTerm, StringComparison.OrdinalIgnoreCase) ||
            r.Author.Contains(searchTerm, StringComparison.OrdinalIgnoreCase) ||
            r.Tags.Any(t => t.Contains(searchTerm, StringComparison.OrdinalIgnoreCase))
        );
        
        OnReviewSearched(searchTerm, results);
        return results;
    }
}

public interface IShareable
{
    void Share();
    void Share(string platform);
}

public class ExplicitShareableReview : Review, IShareable
{
    public string ShareToken { get; set; }
    public bool IsShareEnabled { get; set; }
    
    // НОВЫЕ АТРИБУТЫ
    public Dictionary<string, int> ShareStats { get; private set; }
    public HashSet<string> BlockedPlatforms { get; private set; }
    public DateTime? LastShared { get; private set; }
    
    // НОВЫЕ МЕТОДЫ
    public void UpdateShareStats(string platform)
    {
        if (!ShareStats.ContainsKey(platform))
        {
            ShareStats[platform] = 0;
        }
        ShareStats[platform]++;
        LastShared = DateTime.Now;
    }
    
    public string GetMostPopularPlatform()
    {
        return ShareStats.OrderByDescending(x => x.Value).FirstOrDefault().Key ?? "нет данных";
    }
    
    public void BlockPlatform(string platform)
    {
        BlockedPlatforms.Add(platform.ToLower());
        AddTag($"заблокирована_платформа: {platform}");
    }
    
    public bool IsPlatformBlocked(string platform)
    {
        return BlockedPlatforms.Contains(platform.ToLower());
    }
    
    void IShareable.Share()
    {
        if (IsShareEnabled)
        {
            Console.WriteLine($"[Явная реализация] Отзыв {ReviewId} опубликован с токеном: {ShareToken}");
            IncrementViews();
            UpdateShareStats("default");
        }
        else
        {
            Console.WriteLine("Публикация отключена для этого отзыва");
        }
    }
    
    void IShareable.Share(string platform)
    {
        if (IsShareEnabled && !IsPlatformBlocked(platform))
        {
            Console.WriteLine($"[Явная реализация] Отзыв {ReviewId} опубликован в {platform} с токеном: {ShareToken}");
            IncrementViews();
            AddTag($"явная_публикация: {platform}");
            UpdateShareStats(platform);
        }
        else if (IsPlatformBlocked(platform))
        {
            Console.WriteLine($"Платформа {platform} заблокирована для публикации");
        }
        else
        {
            Console.WriteLine($"Публикация в {platform} отключена");
        }
    }
    
    public void PublicShare()
    {
        Console.WriteLine($"Публичный метод: Отзыв {ReviewId} доступен для общего доступа");
    }
    
    public void GenerateShareToken()
    {
        ShareToken = $"token_{ReviewId}_{DateTime.Now.Ticks}";
        Console.WriteLine($"Сгенерирован токен доступа: {ShareToken}");
    }

    public ExplicitShareableReview(int id, string text, int rating, string author) 
        : base(id, text, rating, author)
    {
        ShareToken = string.Empty;
        IsShareEnabled = true;
        ShareStats = new Dictionary<string, int>();
        BlockedPlatforms = new HashSet<string>();
    }
}

public class LikedReviewsCollection<T> where T : Review
{
    private List<T> _likedReviews = new List<T>();
    private Dictionary<int, T> _likedReviewsDict = new Dictionary<int, T>();
    private IReviewService _reviewService;
    
    // СОБЫТИЯ
    public event Action<T> ReviewLiked;
    public event Action<T> ReviewUnliked;
    
    public LikedReviewsCollection(IReviewService reviewService)
    {
        _reviewService = reviewService ?? throw new ArgumentNullException(nameof(reviewService));
    }
    
    public void AddToLiked(T review)
    {
        if (!review.IsLikedByUser && !_likedReviewsDict.ContainsKey(review.ReviewId))
        {
            review.MarkAsLiked();
            _likedReviews.Add(review);
            _likedReviewsDict[review.ReviewId] = review;
            
            _reviewService.ModerateReview(review);
            ReviewLiked?.Invoke(review);
        }
    }
    
    public void RemoveFromLiked(T review)
    {
        if (_likedReviews.Remove(review) && _likedReviewsDict.Remove(review.ReviewId))
        {
            review.IsLikedByUser = false;
            Console.WriteLine($"Отзыв {review.ReviewId} удален из понравившихся");
            ReviewUnliked?.Invoke(review);
        }
    }
    
    public void RemoveFromLiked(int reviewId)
    {
        if (_likedReviewsDict.TryGetValue(reviewId, out T review))
        {
            RemoveFromLiked(review);
        }
    }
    
    public bool Contains(int reviewId)
    {
        return _likedReviewsDict.ContainsKey(reviewId);
    }
    
    public void PublishAllLikedReviews()
    {
        foreach (var review in _likedReviews)
        {
            _reviewService.PublishReview(review);
        }
    }
    
    public void DisplayLikedReviews()
    {
        if (_likedReviews.Count == 0)
        {
            Console.WriteLine("Пока нет понравившихся отзывов");
            return;
        }
        
        Console.WriteLine($"\nОТЗЫВЫ КОТОРЫЕ ВАМ ПОНРАВИЛИСЬ ({_likedReviews.Count}):");
        Console.WriteLine("==========================================");
        
        foreach (var review in _likedReviews)
        {
            review.DisplayReview();
            review.GetReviewDetails(); 
            Console.WriteLine("------------------------------------------");
        }
    }
    
    public int Count => _likedReviews.Count;
    
    public List<T> GetLikedReviewsByAuthor(string author)
    {
        return _likedReviews.FindAll(r => r.Author.Equals(author, StringComparison.OrdinalIgnoreCase));
    }
    
    // НОВЫЕ МЕТОДЫ
    public Dictionary<string, int> GetLikedReviewsStats()
    {
        var stats = new Dictionary<string, int>();
        foreach (var review in _likedReviews)
        {
            if (!stats.ContainsKey(review.Author))
            {
                stats[review.Author] = 0;
            }
            stats[review.Author]++;
        }
        return stats;
    }
    
    public HashSet<int> GetLikedReviewIds()
    {
        return new HashSet<int>(_likedReviewsDict.Keys);
    }
    
    public void ClearAllLikes()
    {
        foreach (var review in _likedReviews)
        {
            review.IsLikedByUser = false;
        }
        _likedReviews.Clear();
        _likedReviewsDict.Clear();
        Console.WriteLine("Все лайки очищены");
    }
}

public class Review
{
    public int _reviewId;
    public string _text;
    public int _rating;

    public string Author { get; set; }
    public int Likes { get; set; }
    public bool IsVerified { get; set; }
    public DateTime CreationDate { get; set; }
    public List<string> Pros { get; set; }
    public List<string> Cons { get; set; }
    
    public string Language { get; set; }
    public int ViewCount { get; set; }
    public decimal HelpfulnessScore { get; set; }
    public List<string> Tags { get; set; }
    public bool IsLikedByUser { get; set; }
    
    public ReviewStatus Status { get; set; }
    public int ReportCount { get; set; }
    public DateTime LastModified { get; set; }
    public string ReviewHash { get; set; }
    
    // НОВЫЕ АТРИБУТЫ
    public Dictionary<string, object> Metadata { get; private set; }
    public HashSet<string> ViewedBy { get; private set; }
    public List<Review> RelatedReviews { get; private set; }

    public int ReviewId
    {
        get { return _reviewId; }
        set { _reviewId = value; }
    }

    public string Text
    {
        get { return _text; }
        set
        {
            if (string.IsNullOrEmpty(value))
                throw new ArgumentNullException("Отзыв не может быть пустым!");
            _text = value;
            UpdateLastModified();
            GenerateHash();
        }
    }

    public int Rating
    {
        get { return _rating; }
        set
        {
            if (value >= 1 && value <= 5)
                _rating = value;
            else
                throw new ArgumentOutOfRangeException("Рейтинг должен быть от 1 до 5!");
            UpdateLastModified();
        }
    }

    public virtual string ReviewType => "Отзыв о товаре";

    public Review(int id, string text, int rating, string author)
    {
        ReviewId = id;
        Text = text;
        Rating = rating;
        Author = author;
        Likes = 0;
        IsVerified = false;
        CreationDate = DateTime.Now;
        Pros = new List<string>();
        Cons = new List<string>();
        
        Language = "Russian";
        ViewCount = 0;
        HelpfulnessScore = 0;
        Tags = new List<string>();
        IsLikedByUser = false;
        
        Status = ReviewStatus.Pending;
        ReportCount = 0;
        LastModified = DateTime.Now;
        ReviewHash = GenerateHash();
        
        // ИНИЦИАЛИЗАЦИЯ НОВЫХ КОЛЛЕКЦИЙ
        Metadata = new Dictionary<string, object>();
        ViewedBy = new HashSet<string>();
        RelatedReviews = new List<Review>();
    }

    // НОВЫЕ МЕТОДЫ ДЛЯ РАБОТЫ С МЕТАДАННЫМИ
    public void AddMetadata(string key, object value)
    {
        Metadata[key] = value;
        UpdateLastModified();
    }
    
    public T GetMetadata<T>(string key, T defaultValue = default(T))
    {
        if (Metadata.ContainsKey(key) && Metadata[key] is T)
        {
            return (T)Metadata[key];
        }
        return defaultValue;
    }
    
    public void RecordView(string viewer)
    {
        ViewedBy.Add(viewer);
        ViewCount++;
    }
    
    public int GetUniqueViewersCount()
    {
        return ViewedBy.Count;
    }
    
    public void AddRelatedReview(Review relatedReview)
    {
        if (!RelatedReviews.Contains(relatedReview) && relatedReview.ReviewId != this.ReviewId)
        {
            RelatedReviews.Add(relatedReview);
            AddTag($"связан_с_{relatedReview.ReviewId}");
        }
    }
    
    public void DisplayRelatedReviews()
    {
        if (RelatedReviews.Count > 0)
        {
            Console.WriteLine($"Связанные отзывы ({RelatedReviews.Count}):");
            foreach (var related in RelatedReviews)
            {
                Console.WriteLine($"  - ID: {related.ReviewId}, Автор: {related.Author}, Рейтинг: {related.Rating}");
            }
        }
    }

    public virtual void UpdateHelpfulness(bool isHelpful)
    {
        HelpfulnessScore += isHelpful ? 1 : -0.5m;
        UpdateLastModified();
    }
    
    public void AddTag(string tag)
    {
        Tags.Add(tag);
        UpdateLastModified();
    }
    
    public void AddTags(params string[] tags)
    {
        Tags.AddRange(tags);
        UpdateLastModified();
    }
    
    public void IncrementViews()
    {
        ViewCount++;
    }

    // НОВЫЕ МЕТОДЫ
    public void ReportReview(string reason)
    {
        ReportCount++;
        AddTag($"жалоба: {reason}");
        if (ReportCount >= 3)
        {
            Status = ReviewStatus.UnderReview;
            Console.WriteLine($"Отзыв {ReviewId} отправлен на проверку из-за жалоб");
        }
    }
    
    public void UpdateStatus(ReviewStatus newStatus)
    {
        Status = newStatus;
        UpdateLastModified();
        Console.WriteLine($"Статус отзыва {ReviewId} изменен на: {newStatus}");
    }
    
    protected void UpdateLastModified()
    {
        LastModified = DateTime.Now;
    }
    
    public string GenerateHash()
    {
        ReviewHash = $"hash_{ReviewId}_{Text?.GetHashCode()}_{Rating}";
        return ReviewHash;
    }
    
    public virtual bool CanBeEdited()
    {
        return Status != ReviewStatus.Archived && Status != ReviewStatus.Rejected;
    }
    
    public void MarkAsLiked()
    {
        IsLikedByUser = true;
        AddLike();
        AddTag("понравившийся");
        Console.WriteLine($"Отзыв {ReviewId} от {Author} добавлен в понравившиеся!");
    }

    public void AddLike() 
    { 
        Likes++;
        UpdateHelpfulness(true);
        UpdateLastModified();
    }
    
    public void AddPros(params string[] pros) => Pros.AddRange(pros);
    public void AddCons(params string[] cons) => Cons.AddRange(cons);
    
    public void VerifyReview() 
    { 
        IsVerified = true;
        Status = ReviewStatus.Approved;
        UpdateLastModified();
    }
    
    public void EditReview(string newText) 
    { 
        Text = newText;
        UpdateLastModified();
    }
    
    public void EditReview(string newText, string editReason)
    {
        Text = newText;
        AddTag($"редактирован: {editReason}");
        UpdateLastModified();
    }

    public virtual void DisplayReview()
    {
        Console.WriteLine($"[{ReviewType}] ID отзыва: {ReviewId}, Автор: {Author}, Оценка: {Rating}/5");
        Console.WriteLine($"Дата создания: {CreationDate}, Последнее изменение: {LastModified}");
        Console.WriteLine($"Статус: {Status}, Жалоб: {ReportCount}");
        Console.WriteLine($"Подтверждён: {(IsVerified ? "Да" : "Нет")}, Лайков: {Likes}");
        Console.WriteLine($"Полезность: {HelpfulnessScore}, Понравился: {(IsLikedByUser ? "Да" : "Нет")}");
        Console.WriteLine($"Хеш: {ReviewHash}");
        Console.WriteLine($"Уникальные просмотры: {GetUniqueViewersCount()}");
    }

    public virtual void GetReviewDetails()
    {
        Console.WriteLine($"Отзыв: {Text}");
        Console.WriteLine($"Плюсы: {(Pros.Count > 0 ? string.Join(", ", Pros) : "—")}");
        Console.WriteLine($"Минусы: {(Cons.Count > 0 ? string.Join(", ", Cons) : "—")}");
    }

    public void GetReviewDetails(bool includeMetadata)
    {
        if (includeMetadata)
        {
            GetReviewDetails();
            Console.WriteLine($"Просмотры: {ViewCount}, Рейтинг полезности: {HelpfulnessScore}");
            Console.WriteLine($"Статус: {Status}, Жалоб: {ReportCount}");
        }
        else
        {
            GetReviewDetails();
        }
    }

    public void PopularReview(Review popular)
    {
        Console.WriteLine($"Самый популярный отзыв принадлежит пользователю с ID:{popular.ReviewId}:");
    }
}

public class ServiceReview : Review, IShareable
{
    public DateTime VisitDate { get; set; }
    public string EmployeeName { get; set; }
    public string ServiceType { get; set; }
    
    public int ServiceDuration { get; set; }
    public decimal ServiceCost { get; set; }
    public string ServiceLocation { get; set; }
    
    public string ServiceProvider { get; set; }
    public bool AppointmentCompleted { get; set; }
    public decimal TipAmount { get; set; }
    
    // НОВЫЕ АТРИБУТЫ
    public Dictionary<string, int> EmployeeRatings { get; private set; }
    public HashSet<string> ServiceFeatures { get; private set; }
    public List<string> ServicePhotos { get; private set; }

    public override string ReviewType => "Отзыв об услуге";

    public ServiceReview(int id, string text, int rating, string author, DateTime visitDate, string employee, string service)
        : base(id, text, rating, author)
    {
        VisitDate = visitDate;
        EmployeeName = employee;
        ServiceType = service;
        
        ServiceDuration = 0;
        ServiceCost = 0;
        ServiceLocation = "Не указано";
        
        ServiceProvider = "Не указан";
        AppointmentCompleted = true;
        TipAmount = 0;
        
        // ИНИЦИАЛИЗАЦИЯ НОВЫХ КОЛЛЕКЦИЙ
        EmployeeRatings = new Dictionary<string, int>();
        ServiceFeatures = new HashSet<string>();
        ServicePhotos = new List<string>();
    }

    // НОВЫЕ МЕТОДЫ
    public void AddEmployeeRating(string employee, int rating)
    {
        if (rating >= 1 && rating <= 5)
        {
            EmployeeRatings[employee] = rating;
            AddTag($"оценка_сотрудника: {employee}");
        }
    }
    
    public double GetAverageEmployeeRating()
    {
        if (EmployeeRatings.Count == 0) return 0;
        return EmployeeRatings.Values.Average();
    }
    
    public void AddServiceFeature(string feature)
    {
        ServiceFeatures.Add(feature);
        AddTag($"особенность: {feature}");
    }
    
    public void AddServicePhoto(string photoUrl)
    {
        ServicePhotos.Add(photoUrl);
        AddMetadata($"photo_{ServicePhotos.Count}", photoUrl);
    }
    
    public bool HasFeature(string feature)
    {
        return ServiceFeatures.Contains(feature);
    }

    public void SetServiceDetails(int duration, decimal cost, string location)
    {
        ServiceDuration = duration;
        ServiceCost = cost;
        ServiceLocation = location;
        UpdateLastModified();
    }
    
    public void SetProviderInfo(string provider, decimal tip = 0)
    {
        ServiceProvider = provider;
        TipAmount = tip;
        AddTag($"провайдер: {provider}");
    }
    
    public void MarkAppointmentStatus(bool completed)
    {
        AppointmentCompleted = completed;
        if (!completed)
        {
            AddTag("отмена_встречи");
            Status = ReviewStatus.UnderReview;
        }
    }
    
    public decimal CalculateHourlyRate()
    {
        if (ServiceDuration == 0) return 0;
        return ServiceCost / (ServiceDuration / 60m);
    }
    
    public decimal CalculateValueForMoney()
    {
        if (ServiceCost == 0) return 0;
        return (decimal)Rating / ServiceCost * 1000;
    }
    
    public override void UpdateHelpfulness(bool isHelpful)
    {
        base.UpdateHelpfulness(isHelpful);
    }
    
    public override bool CanBeEdited()
    {
        return base.CanBeEdited() && AppointmentCompleted;
    }

    public override void DisplayReview()
    {
        base.DisplayReview();
        Console.WriteLine($"Особенности услуги: {(ServiceFeatures.Count > 0 ? string.Join(", ", ServiceFeatures) : "—")}");
        Console.WriteLine($"Фотографии: {ServicePhotos.Count} шт");
        if (EmployeeRatings.Count > 0)
        {
            Console.WriteLine($"Средняя оценка сотрудников: {GetAverageEmployeeRating():F1}");
        }
    }

    public void Share()
    {
        Console.WriteLine($"Отзыв об услуге '{ServiceType}' опубликован пользователем {Author}!");
        IncrementViews();
        AddTag("поделился");
    }
    
    public void Share(string platform)
    {
        Console.WriteLine($"Отзыв об услуге '{ServiceType}' опубликован в {platform} пользователем {Author}!");
        IncrementViews();
        AddTag($"поделились: {platform}");
    }
}

public class DetailedProductReview : Review, IShareable
{
    public int ProductId { get; set; }
    public string ProductName { get; set; }
    
    public string ProductCategory { get; set; }
    public int UsageDuration { get; set; }
    public bool WouldRecommend { get; set; }
    
    // НОВЫЕ АТРИБУТЫ
    public string PurchaseSource { get; set; }
    public decimal PurchasePrice { get; set; }
    public bool WarrantyActive { get; set; }
    
    // ДОПОЛНИТЕЛЬНЫЕ НОВЫЕ АТРИБУТЫ
    public Dictionary<string, decimal> FeatureScores { get; private set; }
    public HashSet<string> CompatibleProducts { get; private set; }
    public List<DateTime> UsageMilestones { get; private set; }

    public override string ReviewType => "Отзыв о товаре";

    public DetailedProductReview(int id, string text, int rating, string author, int productId, string productName)
        : base(id, text, rating, author)
    {
        ProductId = productId;
        ProductName = productName;
        
        ProductCategory = "Не указана";
        UsageDuration = 0;
        WouldRecommend = rating >= 4;
        
        PurchaseSource = "Не указан";
        PurchasePrice = 0;
        WarrantyActive = false;
        
        // ИНИЦИАЛИЗАЦИЯ НОВЫХ КОЛЛЕКЦИЙ
        FeatureScores = new Dictionary<string, decimal>();
        CompatibleProducts = new HashSet<string>();
        UsageMilestones = new List<DateTime>();
    }

    // НОВЫЕ МЕТОДЫ
    public void AddFeatureScore(string feature, decimal score)
    {
        FeatureScores[feature] = score;
        AddTag($"оценка_характеристики: {feature}");
    }
    
    public decimal GetAverageFeatureScore()
    {
        if (FeatureScores.Count == 0) return 0;
        return FeatureScores.Values.Average();
    }
    
    public void AddCompatibleProduct(string product)
    {
        CompatibleProducts.Add(product);
        AddTag($"совместим_с: {product}");
    }
    
    public void AddUsageMilestone()
    {
        UsageMilestones.Add(DateTime.Now);
        AddTag($"веха_использования_{UsageMilestones.Count}");
    }
    
    public int GetDaysSinceFirstUse()
    {
        if (UsageMilestones.Count == 0) return 0;
        return (int)(DateTime.Now - UsageMilestones.First()).TotalDays;
    }

    public void SetUsageDetails(int durationDays, string category)
    {
        UsageDuration = durationDays;
        ProductCategory = category;
        UpdateLastModified();
    }
    
    // НОВЫЕ МЕТОДЫ
    public void SetPurchaseInfo(string source, decimal price, bool warranty = false)
    {
        PurchaseSource = source;
        PurchasePrice = price;
        WarrantyActive = warranty;
        AddTag($"куплено: {source}");
        
        if (warranty)
        {
            AddTag("гарантия");
        }
    }
    
    public decimal CalculatePriceToRatingRatio()
    {
        if (PurchasePrice == 0) return 0;
        return (decimal)Rating / PurchasePrice;
    }
    
    public void ExtendWarranty(int months)
    {
        WarrantyActive = true;
        AddTag($"гарантия_продлена_{months}мес");
        Console.WriteLine($"Гарантия на {ProductName} продлена на {months} месяцев");
    }
    
    public string GetRecommendationStatus()
    {
        return WouldRecommend ? "Рекомендую к покупке" : "Не рекомендую";
    }
    
    public override void UpdateHelpfulness(bool isHelpful)
    {
        base.UpdateHelpfulness(isHelpful);
    }
    
    public override bool CanBeEdited()
    {
        return base.CanBeEdited() && WarrantyActive;
    }

    public override void DisplayReview()
    {
        base.DisplayReview();
        Console.WriteLine($"Совместимые товары: {(CompatibleProducts.Count > 0 ? string.Join(", ", CompatibleProducts) : "—")}");
        if (FeatureScores.Count > 0)
        {
            Console.WriteLine($"Средняя оценка характеристик: {GetAverageFeatureScore():F1}");
        }
        Console.WriteLine($"Дней использования: {GetDaysSinceFirstUse()}");
    }

    public void Share()
    {
        Console.WriteLine($"Отзыв о товаре '{ProductName}' опубликован пользователем {Author}!");
        IncrementViews();
    }
    
    public void Share(string platform)
    {
        Console.WriteLine($"Отзыв о товаре '{ProductName}' опубликован в {platform} пользователем {Author}!");
        IncrementViews();
        AddTag($"поделились: {platform}");
        if (platform.ToLower().Contains("social"))
        {
            AddLike();
        }
    }
}

public enum ReviewStatus
{
    Pending,
    Approved,
    Rejected,
    UnderReview,
    Archived
}

IReviewService reviewService = new ReviewService();

// ПОДПИСКА НА СОБЫТИЯ
((ReviewService)reviewService).ReviewPublished += (sender, e) => 
{
    Console.WriteLine($"СОБЫТИЕ: Отзыв {e.Review.ReviewId} опубликован в {e.Timestamp}");
};

((ReviewService)reviewService).ReviewModerated += (sender, e) => 
{
    Console.WriteLine($"СОБЫТИЕ: Отзыв {e.Review.ReviewId} прошел модерацию");
};

((ReviewService)reviewService).ReviewSearched += (searchTerm, results) => 
{
    Console.WriteLine($"СОБЫТИЕ: Поиск '{searchTerm}' - найдено {results.Count} отзывов");
};

var likedReviews = new LikedReviewsCollection<Review>(reviewService);

// ПОДПИСКА НА СОБЫТИЯ ЛАЙКОВ
likedReviews.ReviewLiked += (review) => 
{
    Console.WriteLine($"СОБЫТИЕ: Пользователь лайкнул отзыв {review.ReviewId}");
};

likedReviews.ReviewUnliked += (review) => 
{
    Console.WriteLine($"СОБЫТИЕ: Пользователь убрал лайк с отзыва {review.ReviewId}");
};

Console.WriteLine("=== ДЕМОНСТРАЦИЯ ЯВНОЙ РЕАЛИЗАЦИИ ИНТЕРФЕЙСА ===");
var explicitReview = new ExplicitShareableReview(9999, "Тестовый отзыв с явной реализацией", 5, "Тестер");
explicitReview.GenerateShareToken();

// ТЕСТИРУЕМ НОВЫЕ МЕТОДЫ
explicitReview.BlockPlatform("Twitter");
explicitReview.UpdateShareStats("VK");
explicitReview.UpdateShareStats("Telegram");
explicitReview.UpdateShareStats("VK");

Console.WriteLine($"Самая популярная платформа: {explicitReview.GetMostPopularPlatform()}");

IShareable shareable = explicitReview;
shareable.Share();
shareable.Share("Telegram");
shareable.Share("Twitter"); // Должен быть заблокирован

explicitReview.PublicShare();
Console.WriteLine();

DateTime reviewDate1 = new DateTime(2025, 9, 22, 1, 43, 59);
var serviceReview1 = new ServiceReview(1703, "Сотрудник помог с выбором товара", 5, "Анна", reviewDate1, "Павел", "Консультация");
var productReview1 = new DetailedProductReview(1703, "Восхитительно! Товар полностью оправдал ожидания.", 5, "Анна", 123, "Холодильник Samsung");


serviceReview1.AddEmployeeRating("Павел", 5);
serviceReview1.AddEmployeeRating("Менеджер", 4);
serviceReview1.AddServiceFeature("быстрая консультация");
serviceReview1.AddServiceFeature("профессионализм");
serviceReview1.AddServicePhoto("https://example.com/photo1.jpg");

productReview1.AddFeatureScore("качество", 4.8m);
productReview1.AddFeatureScore("дизайн", 4.5m);
productReview1.AddFeatureScore("функциональность", 4.9m);
productReview1.AddCompatibleProduct("Смартфон Samsung");
productReview1.AddCompatibleProduct("Планшет Samsung");
productReview1.AddUsageMilestone();
productReview1.AddUsageMilestone();

serviceReview1.SetServiceDetails(30, 1500, "Москва, ТЦ Авиапарк");
serviceReview1.SetProviderInfo("ООО 'СервисПлюс'", 200);
serviceReview1.AddTags("консультация", "помощь");

productReview1.SetUsageDetails(45, "Бытовая техника");
productReview1.SetPurchaseInfo("DNS", 45000, true);
productReview1.AddTags("качество", "доставка", "холодильник");
productReview1.WouldRecommend = true;

serviceReview1.AddPros("Вежливость", "Компетентность");
serviceReview1.AddCons();
productReview1.AddPros("Качество", "Быстрая доставка");
productReview1.AddCons("Высокая цена");
serviceReview1.VerifyReview();
productReview1.VerifyReview();

serviceReview1.MarkAppointmentStatus(true);
productReview1.ExtendWarranty(12);

likedReviews.AddToLiked(serviceReview1);
likedReviews.AddToLiked(productReview1);

reviewService.PublishReview(serviceReview1);
reviewService.PublishReview(productReview1);

Console.WriteLine();
productReview1.PopularReview(productReview1);
Console.WriteLine();

Console.WriteLine("=== ОТЗЫВ ПОЛЬЗОВАТЕЛЯ АННА ===");
serviceReview1.DisplayReview();
serviceReview1.GetReviewDetails();
Console.WriteLine();
productReview1.DisplayReview();
productReview1.GetReviewDetails();

serviceReview1.Share();
serviceReview1.Share("ВКонтакте");
productReview1.Share("Twitter");

Console.WriteLine("\n=== СИСТЕМА ЖАЛОБ ===");
productReview1.ReportReview("некорректная информация");
productReview1.ReportReview("спам");
productReview1.ReportReview("оскорбления");

Console.WriteLine();
Console.WriteLine("----------------------------------------------------------------------------------------");
Console.WriteLine();

// ТЕСТИРУЕМ НОВЫЕ ВОЗМОЖНОСТИ СЕРВИСА"0)))))))"
Console.WriteLine("=== ТАК НАЗЫВАЕМЫЕ НОВЫЕ ВОЗМОЖНОСТИ СЕРВИСА ===");

var authorStats = reviewService.GetAuthorStats();
Console.WriteLine("Статистика авторов:");
foreach (var stat in authorStats)
{
    Console.WriteLine($"  {stat.Key}: {stat.Value} отзывов");
}

var allTags = reviewService.GetAllTags();
Console.WriteLine($"Всего уникальных тегов: {allTags.Count}");

var searchResults = reviewService.SearchReviews("качество");
Console.WriteLine($"Найдено отзывов по запросу 'качество': {searchResults.Count}");

var likedStats = likedReviews.GetLikedReviewsStats();
Console.WriteLine("Статистика лайков по авторам:");
foreach (var stat in likedStats)
{
    Console.WriteLine($"  {stat.Key}: {stat.Value} лайков");
}

Console.WriteLine("----------------------------------------------------------------------------------------");
Console.WriteLine();

DateTime reviewDate2 = new DateTime(2025, 10, 5, 10, 25, 12);
var serviceReview2 = new ServiceReview(5080, "Сотрудник был груб и не помог с выбором.", 1, "Олег", reviewDate2, "Марина", "Консультация");
var productReview2 = new DetailedProductReview(5080, "Отказался от покупки товара: плохое качество.", 1, "Олег", 321, "Смартфон Poco X7 Pro");

serviceReview2.SetServiceDetails(15, 500, "Санкт-Петербург");
serviceReview2.MarkAppointmentStatus(false); // Встреча не завершена
productReview2.SetUsageDetails(2, "Электроника");
productReview2.SetPurchaseInfo("ТРЦ Колумб", 15000, false);

serviceReview2.AddPros("Их тупо нет!");
serviceReview2.AddCons("Грубость", "Неопытность");
productReview2.AddPros();
productReview2.AddCons("Плохая сборка", "Слабая батарея");

Console.WriteLine("=== ОТЗЫВ ПОЛЬЗОВАТЕЛЯ ОЛЕГ ===");
serviceReview2.DisplayReview();
serviceReview2.GetReviewDetails();
Console.WriteLine();
productReview2.DisplayReview();
productReview2.GetReviewDetails();

Console.WriteLine();
Console.WriteLine("----------------------------------------------------------------------------------------");
Console.WriteLine();

DateTime reviewDate3 = new DateTime(2025, 10, 5, 15, 40, 00);
var serviceReview3 = new ServiceReview(9020, "Мастер быстро починил технику.", 5, "Виктор", reviewDate3, "Игорь", "Ремонт техники");
var productReview3 = new DetailedProductReview(9020, "Хороший ноутбук, но греется при нагрузке.", 4, "Виктор", 456, "ASUS ZenBook");

serviceReview3.SetServiceDetails(120, 3000, "Казань");
serviceReview3.SetProviderInfo("СЦ 'Компьютерный доктор'", 300);
productReview3.SetUsageDetails(60, "Компьютеры");
productReview3.SetPurchaseInfo("М.Видео", 75000, true);

productReview3.AddPros("Производительность", "Качество сборки");
productReview3.AddCons("Перегрев", "Цена");
serviceReview3.VerifyReview();
productReview3.VerifyReview();

likedReviews.AddToLiked(serviceReview3);
likedReviews.AddToLiked(productReview3);

likedReviews.PublishAllLikedReviews();

Console.WriteLine();

Console.WriteLine("=== ОТЗЫВ ПОЛЬЗОВАТЕЛЯ ВИКТОР ===");
serviceReview3.DisplayReview();
serviceReview3.GetReviewDetails();
Console.WriteLine();
productReview3.DisplayReview();
productReview3.GetReviewDetails();

Console.WriteLine("\n=== ПОПУЛЯРНЫЕ ОТЗЫВЫ (через сервис) ===");
var popularReviews = reviewService.GetPopularReviews(2);
Console.WriteLine($"Найдено популярных отзывов: {popularReviews.Count}");

likedReviews.DisplayLikedReviews();

Console.WriteLine();
Console.WriteLine("----------------------------------------------------------------------------------------");


=== ДЕМОНСТРАЦИЯ ЯВНОЙ РЕАЛИЗАЦИИ ИНТЕРФЕЙСА ===
Сгенерирован токен доступа: token_9999_638989340536961763
Самая популярная платформа: VK
[Явная реализация] Отзыв 9999 опубликован с токеном: token_9999_638989340536961763
[Явная реализация] Отзыв 9999 опубликован в Telegram с токеном: token_9999_638989340536961763
Платформа Twitter заблокирована для публикации
Публичный метод: Отзыв 9999 доступен для общего доступа

Гарантия на Холодильник Samsung продлена на 12 месяцев
Отзыв 1703 от Анна добавлен в понравившиеся!
Отзыв 1703 прошел модерацию
СОБЫТИЕ: Отзыв 1703 прошел модерацию
СОБЫТИЕ: Пользователь лайкнул отзыв 1703
Отзыв 1703 опубликован в системе
СОБЫТИЕ: Отзыв 1703 опубликован в 11/16/2025 11:54:13 PM
Отзыв 1703 опубликован в системе
СОБЫТИЕ: Отзыв 1703 опубликован в 11/16/2025 11:54:13 PM

Самый популярный отзыв принадлежит пользователю с ID:1703:

=== ОТЗЫВ ПОЛЬЗОВАТЕЛЯ АННА ===
[Отзыв об услуге] ID отзыва: 1703, Автор: Анна, Оценка: 5/5
Дата создания: 11/16/2025 11:54:13 PM, Пос